# Patches — Bloc 6, Sessió 11
### FFT pràctic i extracció de features espectrals

**Com obrir aquest notebook:** Fes clic a l'enllaç del Classroom. Per guardar els teus canvis: `File → Save a copy in Drive`.

Aquesta sessió té dues parts ben diferenciades:
1. **FFT manual** amb `np.fft.rfft()` — per entendre què és realment una transformada de Fourier, sense cap llibreria d'àudio.
2. **Features espectrals amb `librosa`** (centroid, ZCR, MFCC) — eines ja construïdes i optimitzades, perquè reinventar-les seria una pèrdua de temps per al que volem aconseguir avui: preparar dades per a un classificador (Sessió 12).

**Primer pas:** clona el repositori del curs (o puja la carpeta `dataset/` si treballes fora de l'entorn del Classroom) perquè aquest notebook hi pugui accedir.

In [ ]:
# Si treballes des de Google Colab amb el repositori clonat:
!git clone -q https://github.com/jcomajuncosas/tp2.git /content/tp2 2>/dev/null || echo "Ja clonat o no disponible"
DATASET_DIR = "/content/tp2/06_analisi_ml/sessio11_fft_features/dataset"

# Si el clone no funciona (per exemple, repositori privat), puja manualment
# la carpeta 'dataset/' (kick/, hihat/, CREDITS.md) i ajusta aquesta ruta:
# DATASET_DIR = "/content/dataset"

import os
print("Existeix?", os.path.exists(DATASET_DIR))
print("Kicks:", len(os.listdir(f"{DATASET_DIR}/kick")) if os.path.exists(DATASET_DIR) else "N/A")
print("Hihats:", len(os.listdir(f"{DATASET_DIR}/hihat")) if os.path.exists(DATASET_DIR) else "N/A")

In [ ]:
import numpy as np
import librosa
import librosa.display
import matplotlib.pyplot as plt
from IPython.display import Audio, display
import glob

sample_rate_target = None  # mantenim el sample rate original de cada fitxer (16kHz)

## 1. FFT manual: què hi ha realment dins d'un so?

Recordeu el Bloc 1: un so és un array de números (amplitud al llarg del temps). Una FFT (*Fast Fourier Transform*) el transforma en un altre array: quanta energia hi ha a cada freqüència.

`np.fft.rfft()` ja n'hi ha prou — no necessitem cap llibreria d'àudio per a la matemàtica en si.

In [ ]:
# Carreguem un kick i un hihat per comparar
kick_files = sorted(glob.glob(f"{DATASET_DIR}/kick/*.wav"))
hihat_files = sorted(glob.glob(f"{DATASET_DIR}/hihat/*.wav"))

y_kick, sr_kick = librosa.load(kick_files[0], sr=None)
y_hihat, sr_hihat = librosa.load(hihat_files[0], sr=None)

print(f"Kick: {kick_files[0].split('/')[-1]}, {len(y_kick)} mostres a {sr_kick}Hz")
print(f"Hihat: {hihat_files[0].split('/')[-1]}, {len(y_hihat)} mostres a {sr_hihat}Hz")

print("\nKick:")
display(Audio(y_kick, rate=sr_kick))
print("Hihat:")
display(Audio(y_hihat, rate=sr_hihat))

In [ ]:
# FFT manual amb NumPy
fft_kick = np.fft.rfft(y_kick)
freqs_kick = np.fft.rfftfreq(len(y_kick), 1/sr_kick)
magnitude_kick = np.abs(fft_kick)

fft_hihat = np.fft.rfft(y_hihat)
freqs_hihat = np.fft.rfftfreq(len(y_hihat), 1/sr_hihat)
magnitude_hihat = np.abs(fft_hihat)

fig, axs = plt.subplots(1, 2, figsize=(12, 4))
axs[0].plot(freqs_kick, magnitude_kick)
axs[0].set_title("Espectre del KICK")
axs[0].set_xlabel("Freqüència (Hz)")
axs[0].set_xlim(0, 2000)

axs[1].plot(freqs_hihat, magnitude_hihat, color='orange')
axs[1].set_title("Espectre del HIHAT")
axs[1].set_xlabel("Freqüència (Hz)")
axs[1].set_xlim(0, 2000)

plt.tight_layout()
plt.show()

peak_kick = freqs_kick[np.argmax(magnitude_kick)]
peak_hihat = freqs_hihat[np.argmax(magnitude_hihat)]
print(f"Freqüència de pic del kick:  {peak_kick:.0f} Hz")
print(f"Freqüència de pic del hihat: {peak_hihat:.0f} Hz")
print()
print("El kick concentra l'energia en freqüències baixes (mireu el gràfic:")
print("pràcticament tot per sota de 200Hz). El hihat reparteix l'energia per")
print("un rang molt més ampli i agut — per això un únic 'pic' no el descriu")
print("tan bé com en el cas del kick; necessitem una mesura millor, com el")
print("centroide espectral que veurem ara.")

## 2. Per què NO implementem les features a mà

Un cop hem vist què és la FFT, el pas natural seria calcular el "centroide espectral" (la freqüència mitjana ponderada per energia) nosaltres mateixos. Però aquestes features ja estan ben implementades, testejades i optimitzades a `librosa` — reinventar-les no ensenya res de nou que la FFT manual ja no hagi ensenyat, i ens fa perdre temps que necessitem per a la part important: preparar dades reals per al classificador de la propera sessió.

**Per això avui passem directament a `librosa` per a les features.**

In [ ]:
# Centroide espectral: "freqüència mitjana" del so, ponderada per energia
centroid_kick = librosa.feature.spectral_centroid(y=y_kick, sr=sr_kick).mean()
centroid_hihat = librosa.feature.spectral_centroid(y=y_hihat, sr=sr_hihat).mean()

print(f"Centroide espectral del kick:  {centroid_kick:.0f} Hz")
print(f"Centroide espectral del hihat: {centroid_hihat:.0f} Hz")
print("\n(Confirma numèricament el que vèiem als gràfics de la FFT manual)")

In [ ]:
# Zero Crossing Rate (ZCR): quants cops el senyal travessa zero per segon
# Sons sorollosos/percussius aguts (com un hihat) tenen ZCR alt
zcr_kick = librosa.feature.zero_crossing_rate(y_kick).mean()
zcr_hihat = librosa.feature.zero_crossing_rate(y_hihat).mean()

print(f"ZCR del kick:  {zcr_kick:.4f}")
print(f"ZCR del hihat: {zcr_hihat:.4f}")

In [ ]:
# MFCC (Mel-Frequency Cepstral Coefficients): una representació compacta
# del timbre, molt usada en reconeixement de veu i classificació de so
mfcc_kick = librosa.feature.mfcc(y=y_kick, sr=sr_kick, n_mfcc=13)
mfcc_hihat = librosa.feature.mfcc(y=y_hihat, sr=sr_hihat, n_mfcc=13)

fig, axs = plt.subplots(1, 2, figsize=(12, 4))
librosa.display.specshow(mfcc_kick, x_axis='time', ax=axs[0])
axs[0].set_title("MFCC del KICK")
librosa.display.specshow(mfcc_hihat, x_axis='time', ax=axs[1])
axs[1].set_title("MFCC del HIHAT")
plt.tight_layout()
plt.show()

print("13 coeficients MFCC, un valor per cada 'banda' tímbrica del so.")
print("No cal entendre la matemàtica exacta (bancs de filtres Mel, DCT...)")
print("— el que importa avui és que és una manera estàndard i compacta de")
print("descriure el timbre d'un so amb pocs números.")

## 3. Construint el mini-dataset per a la Sessió 12

Ara extraiem aquestes tres features (centroid, ZCR, MFCC) de TOTS els sons del mini-dataset (14 kicks + 15 hihats), i les guardem en una taula — exactament el format que necessitarem per entrenar un classificador la propera sessió.

In [ ]:
import warnings
warnings.filterwarnings('ignore')  # suprimeix avisos benignes de n_fft en sons curts

files_and_labels = (
    [(f, 'kick') for f in kick_files] +
    [(f, 'hihat') for f in hihat_files]
)

rows = []
for filepath, label in files_and_labels:
    y, sr = librosa.load(filepath, sr=None)
    if len(y) < 256:
        continue
    n_fft = min(2048, len(y))  # evita l'avís quan el so és molt curt

    centroid = librosa.feature.spectral_centroid(y=y, sr=sr, n_fft=n_fft).mean()
    zcr = librosa.feature.zero_crossing_rate(y).mean()
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13, n_fft=n_fft).mean(axis=1)

    row = {
        'fitxer': filepath.split('/')[-1],
        'classe': label,
        'centroid': centroid,
        'zcr': zcr,
    }
    for i, val in enumerate(mfcc):
        row[f'mfcc_{i}'] = val
    rows.append(row)

print(f"Dataset construït: {len(rows)} sons, {len(rows[0]) - 2} features cadascun")

In [ ]:
import pandas as pd

df = pd.DataFrame(rows)
df.head(10)

In [ ]:
# Visualitzem la separació entre classes amb només dues features
fig, ax = plt.subplots(figsize=(7, 5))
for label, color in [('kick', '#2a9d8f'), ('hihat', '#e76f51')]:
    subset = df[df['classe'] == label]
    ax.scatter(subset['centroid'], subset['zcr'], label=label, color=color, s=80)

ax.set_xlabel('Centroide espectral (Hz)')
ax.set_ylabel('Zero Crossing Rate')
ax.set_title('Kick vs. Hihat: separació amb només 2 features')
ax.legend()
plt.show()

print("Amb només centroid i ZCR ja es veu una separació molt clara —")
print("la propera sessió un classificador aprendrà aquesta frontera automàticament.")

In [ ]:
# Guardem el dataset com a CSV per a la Sessió 12
df.to_csv('features_percussio.csv', index=False)
print("Desat a features_percussio.csv")
print(f"Forma final: {df.shape[0]} files, {df.shape[1]} columnes")

## Recordatori sobre el dataset

Els 29 sons d'aquest mini-dataset provenen del **Freesound One-Shot Percussive Sounds Dataset** (Ramires et al., Music Technology Group - UPF, DOI: [10.5281/zenodo.3665275](https://doi.org/10.5281/zenodo.3665275)), filtrats per llicència CC0/CC-BY 3.0. Atribució completa per fitxer a `dataset/CREDITS.md`.

## Per provar tu mateix

Canvia quins fitxers carregues a la Secció 1, o prova de calcular una quarta feature (per exemple `librosa.feature.spectral_bandwidth` o `librosa.feature.rms`) i afegeix-la al DataFrame.

## Preview Sessió 12

Amb `features_percussio.csv` ja preparat, la propera sessió entrenarem un classificador (KNN o arbre de decisió) que aprengui a distingir kick de hihat a partir d'aquestes features — el primer pas cap a ML real amb dades d'àudio.